In [0]:
Bronze_Schema_Path = "assesment_pipline.bronze"

In [0]:
dbutils.widgets.text("valid_records_table_name", "")
valid_records_table_name = dbutils.widgets.get("valid_records_table_name")

In [0]:
expected_silver_count = dbutils.jobs.taskValues.get(
    taskKey="Bronze_Creation",
    key="expected_silver_count",
    debugValue=0
)

In [0]:
valied_bronze_df=spark.read.table(f"{Bronze_Schema_Path}.{valid_records_table_name}")
display(valied_bronze_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,4,375.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,5,550.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,6,725.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,7,900.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,Hyderabad,2,1075.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,Pune,3,1250.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,Mumbai,4,1425.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,Kochi,5,200.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,Delhi,1,375.0,20.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,Coimbatore,2,550.0,40.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z


In [0]:
from pyspark.sql.functions import trim,upper,col
silver_df=valied_bronze_df.withColumn("product_name",trim(col("product_name"))).withColumn("category",upper(trim(col("category")))).withColumn("shipping_city",upper(trim(col("shipping_city")))).withColumn("net_order_amount",((col("quantity")*col("unit_price"))-col("discount")))
display(silver_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp,net_order_amount
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,DELHI,4,375.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,1480.0
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,COIMBATORE,5,550.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,2710.0
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,BENGALURU,6,725.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,4290.0
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,CHENNAI,7,900.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,6300.0
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,HYDERABAD,2,1075.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,2130.0
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,PUNE,3,1250.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,3710.0
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,MUMBAI,4,1425.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,5640.0
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,KOCHI,5,200.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,1000.0
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,DELHI,1,375.0,20.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,355.0
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,COIMBATORE,2,550.0,40.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:10.712Z,1060.0


In [0]:
actual_silver_count = silver_df.count()
print("Actual Silver Count:", actual_silver_count)
print("Expected Silver Count:",expected_silver_count)

if(actual_silver_count == expected_silver_count):
  print("Reconciliation PASS")
else:
  print("Reconciliation FAILED")

Actual Silver Count: 12
Expected Silver Count: 0
Reconciliation FAILED


# Start of Merge

In [0]:
from delta.tables import DeltaTable

silver_delta=DeltaTable.forName(spark,"assesment_pipline.silver.silver_orders")

target_silver_df=silver_delta.toDF()
silver_count_before_merge=target_silver_df.count()

display(target_silver_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp,net_order_amount
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,DELHI,2,375,20,PLACED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,730
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,COIMBATORE,3,550,40,DELIVERED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,1610
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,BENGALURU,4,725,60,CANCELLED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,2840
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,CHENNAI,5,900,0,PLACED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,4500
ORD0005,2026-10-11,C005,P005,Product 6,BEVERAGES,HYDERABAD,1,1075,20,DELIVERED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,1055
ORD0006,2026-10-12,C006,P006,Product 7,HOUSEHOLD,PUNE,2,1250,40,CANCELLED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,2460
ORD0007,2026-10-13,C007,P007,Product 8,GROCERY,MUMBAI,3,1425,60,PLACED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,4215
ORD0008,2026-10-14,C008,P008,Product 1,BEVERAGES,KOCHI,4,200,0,DELIVERED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,800
ORD0009,2026-10-15,C009,P009,Product 2,HOUSEHOLD,DELHI,5,375,20,CANCELLED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,1855
ORD0010,2026-10-16,C010,P010,Product 3,GROCERY,COIMBATORE,1,550,40,PLACED,BATCH_001,online_shopping_initial.csv,2026-11-23T07:13:08.574Z,510


In [0]:
def check_df_compatibility(target_df, source_df):
   
    target_schema = {
        field.name: field.dataType.simpleString()
        for field in target_df.schema.fields
    }
   
    source_schema = {
        field.name: field.dataType.simpleString()
        for field in source_df.schema.fields
    }
    if target_schema == source_schema:
        print("PASS: Both DataFrames are compatible for MERGE")
        return True
    else:
        print("FAIL: DataFrames are not compatible")
 
        print("\nTarget Schema:")
        print(target_schema)
 
        print("\nSource Schema:")
        print(source_schema)
 
        return False
    
check_df_compatibility(
    target_silver_df,
    silver_df
)

PASS: Both DataFrames are compatible for MERGE


True

In [0]:
new_records=silver_df.join(target_silver_df,["order_id"],how="left_anti")
new_records_count=new_records.count()
print("New Records: ",new_records_count)

New Records:  8


In [0]:
update_records=silver_df.join(target_silver_df,["order_id"],how="inner")
update_records_count=update_records.count()
print("Records Thats need to be Updated",update_records_count)

Records Thats need to be Updated 4


In [0]:
expected_silver_count=silver_count_before_merge+new_records_count
print("Silver Count Before Merge: ",silver_count_before_merge)
print("New Records: ",new_records_count)
print("Records Thats need to be Updated",update_records_count)
print("Expected Silve Count After Merge: ",expected_silver_count)

Silver Count Before Merge:  36
New Records:  8
Records Thats need to be Updated 4
Expected Silve Count After Merge:  44


In [0]:
(
    silver_delta.alias("t").merge(silver_df.alias("s"),"t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
from delta.tables import DeltaTable

silver_delta=DeltaTable.forName(spark,"assesment_pipline.silver.silver_orders")

silver_df=silver_delta.toDF()
display(silver_df)

In [0]:
actual_silver_count = silver_df.count()
print("Actual Silver Count: ", actual_silver_count)
print("Expected Silver Count: ",expected_silver_count)

if(actual_silver_count == expected_silver_count):
  print("Reconciliation PASS")
else:
  print("Reconciliation FAILED")

Actual Silver Count: 44
Reconciliation PASS
